# 각인(imprint) 인식 모델 - Colab 학습

**이 노트북을 실행하기 전에 로컬/서버에서 먼저 해두셔야 하는 것:**

1. 로컬 PC에서 압축:
   ```powershell
   Compress-Archive -Path ML\IMPRINT\data\crops,ML\IMPRINT\data\labels.csv,ML\IMPRINT\train_imprint_model.py -DestinationPath imprint_dataset.zip
   ```
2. AWS 서버로 업로드:
   ```bash
   scp imprint_dataset.zip bomul@<서버IP>:~/imprint_dataset.zip
   ```
3. 서버에서 임시 HTTP 서빙 (+ AWS 보안그룹에 아래에서 쓸 포트 인바운드 규칙 임시 추가):
   ```bash
   python3 -m http.server 8899
   ```

다 되셨으면 아래 순서대로 실행하세요. **런타임 → 런타임 유형 변경 → GPU(T4)** 먼저 선택해두는 것도 잊지 마세요.

## 0. GPU 확인

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU가 안 잡혔습니다 - 런타임 유형을 GPU로 바꾸고 다시 연결하세요.")

서버 퍼블릭 IP(54.180.81.50)에 8899 포트를 붙여서 씁니다.
http.server는 nginx를 거치지 않는 별도 포트라, AWS 보안그룹에서 8899 인바운드를 열어야 접근됩니다.

In [ ]:
SERVER_URL = "http://54.180.81.50:8899/imprint_dataset.zip"

!wget -q --show-progress "$SERVER_URL" -O imprint_dataset.zip
!ls -lh imprint_dataset.zip

## 2. 압축 해제

In [ ]:
!mkdir -p /content/imprint_ml
!unzip -q imprint_dataset.zip -d /content/imprint_ml
%cd /content/imprint_ml
!ls -la
!echo "---"
!ls data | head
!echo "---"
!wc -l data/labels.csv

## 3. 의존성 확인

Colab엔 보통 torch/numpy/opencv가 기본 설치돼 있습니다. 혹시 `cv2` import 에러가 나면 아래 셀의 주석을 풀고 실행하세요.

In [ ]:
# !pip install -q opencv-python-headless

import cv2
import numpy as np
import torch
print("cv2", cv2.__version__, "/ torch", torch.__version__)

## 4. 학습 실행

GPU 메모리 여유가 있으니 `--batch-size`는 로컬(CPU)보다 올려서 씁니다. 필요하면 `--epochs`, `--lr`도 조정하세요.

In [ ]:
!python train_imprint_model.py --epochs 50 --batch-size 64

## 5. 학습된 모델 다운로드

`ML/IMPRINT/models/imprint_crnn.pt`로 저장됩니다(에폭마다 검증 정확도가 갱신될 때만 저장). 로컬로 받아가세요.

In [ ]:
from google.colab import files
files.download("models/imprint_crnn.pt")

## 6. (중요) 다 받았으면 서버 정리

이 노트북에서 실행하는 게 아니라 **서버(AWS)에 직접 접속해서** 하셔야 합니다:

1. `http.server` 띄워둔 터미널에서 `Ctrl+C`로 종료
2. `rm ~/imprint_dataset.zip` (여유 공간 확보)
3. **AWS 보안그룹에서 8899 인바운드 규칙 삭제** — 이거 꼭 하세요, 안 지우면 서버가 계속 외부에 노출된 채로 남습니다